# Experiment: Donor Rejuvenation

## 목적
기증 난자 사용 시 환자 나이가 아니라 기증자 나이 기반 feature가 효과 있는지 확인

## 추가 feature
- 난자_기준나이
- 기증난자_사용여부
- 고령환자_기증난자
- 난자기준_고령여부
- rejuv_gap_numeric
- 기증자_최적연령여부

## 기준 점수
기존 best FE + Cat 단일 valid: 0.737416

## 결과
roc_auc:
f1:
precision:
recall:

## 판단
keep / reject

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import rankdata

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

import lightgbm as lgb
from lightgbm import LGBMClassifier

from category_encoders import TargetEncoder

PROJECT_ROOT = Path("/mnt/c/dev/my_ml_project")

DATA_DIR = PROJECT_ROOT / "data"
OOF_DIR = PROJECT_ROOT / "oof_preds"
SUB_DIR = PROJECT_ROOT / "submissions"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

TARGET = "임신 성공 여부"

FOLD_SEED = 42
N_SPLITS = 5
POS_WEIGHT = 190123 / 66228

SAVE_DIR = OOF_DIR / "combo_te_v1_lgbm_seed42"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

LGBM_SUB_DIR = SUB_DIR / "lgbm_blend"
LGBM_SUB_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_PATH exists:", TRAIN_PATH.exists())
print("SAVE_DIR:", SAVE_DIR)

def data_preprocessing(df):
    df = df.copy()
    time_cols = [
        '임신 시도 또는 마지막 임신 경과 연수',
        '난자 해동 경과일',
        '난자 혼합 경과일',
        '배아 이식 경과일',
        '배아 해동 경과일'
    ]

    for col in time_cols:
        df[f'{col}_performed'] = (
            df[col].notnull()
        ).astype(int)


    df = df.fillna(0)
    infertility_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 남성 요인',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    male_cols = [
        '불임 원인 - 남성 요인',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    female_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증'
    ]

    df['불임원인_총개수'] = df[infertility_cols].sum(axis=1)

    df['남성_원인_수'] = df[male_cols].sum(axis=1)

    df['여성_원인_수'] = df[female_cols].sum(axis=1)

    df['남녀_복합_원인'] = (
        (df['남성_원인_수'] > 0) &
        (df['여성_원인_수'] > 0)
    ).astype(int)

    df['원인불명'] = (
        df['불임원인_총개수'] == 0
    ).astype(int)

    count_cols = [
        '총 시술 횟수',
        'IVF 시술 횟수',
        'DI 시술 횟수',
        '총 임신 횟수',
        'IVF 임신 횟수',
        'DI 임신 횟수',
        '총 출산 횟수',
        'IVF 출산 횟수',
        'DI 출산 횟수',
        '클리닉 내 총 시술 횟수'
    ]

    count_map = {
        '0회': 0,
        '1회': 1,
        '2회': 2,
        '3회': 3,
        '4회': 4,
        '5회': 5,
        '6회 이상': 6,
    }

    for col in count_cols:
        df[col] = df[col].map(count_map).astype(float)

    df['고령여부'] = df['시술 당시 나이'].isin([
        '만38-39세',
        '만40-42세',
        '만43-44세',
        '만45-50세'
    ]).astype(int)


    df['배아_생성률'] = np.where(
        df['혼합된 난자 수'] == 0,
        0,
        df['총 생성 배아 수'] / df['혼합된 난자 수']
    )

    # 2. 배아 이식 효율
    df['배아_이식률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['이식된 배아 수'] / df['총 생성 배아 수']
    )

    # 3. 배아 냉동 비율
    df['배아_냉동률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['저장된 배아 수'] / df['총 생성 배아 수']
    )
    df['IVF_임신성공률'] = np.where(
        df['IVF 시술 횟수'] == 0,
        0,
        df['IVF 임신 횟수'] / df['IVF 시술 횟수']
    )

    df['DI_임신성공률'] = np.where(
        df['DI 시술 횟수'] == 0,
        0,
        df['DI 임신 횟수'] / df['DI 시술 횟수']
    )

    df['고령_난자수_interaction'] = (
        df['고령여부'] *
        df['수집된 신선 난자 수']
    )

    df['배아이식_수행여부'] = (
        df['이식된 배아 수'] > 0
    ).astype(int)

    df['배아_이식_집중도'] = np.where(
        (df['이식된 배아 수'] + df['저장된 배아 수']) == 0,
        0,
        df['이식된 배아 수'] /
        (
            df['이식된 배아 수'] +
            df['저장된 배아 수']
        )
    )
    df["배아 생성 주요 이유"] = (
        df["배아 생성 주요 이유"]
        .astype(str)
        .astype("category")
    )
    df["특정 시술 유형"] = (
        df["특정 시술 유형"]
        .astype(str)
        .astype("category")
    )

    # 고령 × 이식 배아 수
    df['고령_배아이식'] = (
        df['고령여부'] *
        df['이식된 배아 수']
    )

    # 고령 × 총 생성 배아 수
    df['고령_배아생성'] = (
        df['고령여부'] *
        df['총 생성 배아 수']
    )

    # 고령 × 저장 배아 수
    df['고령_배아저장'] = (
        df['고령여부'] *
        df['저장된 배아 수']
    )

    # 고령 × 미세주입 난자 수
    df['고령_미세주입난자'] = (
        df['고령여부'] *
        df['미세주입된 난자 수']
    )

    df['출산_임신_전환율'] = np.where(
        df['총 임신 횟수'] == 0,
        0,
        df['총 출산 횟수'] / df['총 임신 횟수']
    )

    df['클리닉_집중도'] = np.where(
        df['총 시술 횟수'] == 0,
        0,
        df['클리닉 내 총 시술 횟수'] / df['총 시술 횟수']
    )

    df['첫_시술_여부'] = (
        df['총 시술 횟수'] == 0
    ).astype(int)



    binary_keywords = [
        "코드", "나이", "유형", "여부", "원인", "이유", "횟수", "출처"
    ]

    binary_cols = [
        col for col in df.columns
        if any(keyword in col for keyword in binary_keywords)
    ]

    df[binary_cols] = df[binary_cols].astype('category')

    object_cols = df.select_dtypes(include="object").columns

    df[object_cols] = df[object_cols].astype(str)

    cat_cols = df.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    for col in cat_cols:
        if col != TARGET:
            df[col] = df[col].astype(str)

    drop_cols = [
        '배아이식_수행여부'
    ]
    df = df.drop(
        columns=drop_cols
        )
    return df

PROJECT_ROOT: /mnt/c/dev/my_ml_project
TRAIN_PATH exists: True
SAVE_DIR: /mnt/c/dev/my_ml_project/oof_preds/combo_te_v1_lgbm_seed42


In [3]:
def rank01(pred):
    return rankdata(pred) / len(pred)


def add_combo_columns(X):
    X = X.copy()

    combo_pairs = [
        ("시술 당시 나이", "난자 출처"),
        ("시술 당시 나이", "정자 출처"),
        ("시술 당시 나이", "시술 유형"),
        ("시술 당시 나이", "특정 시술 유형"),
        ("시술 유형", "난자 출처"),
        ("시술 유형", "정자 출처"),
        ("특정 시술 유형", "난자 출처"),
        ("특정 시술 유형", "정자 출처"),
        ("난자 출처", "정자 출처"),
        ("배란 유도 유형", "시술 당시 나이"),
    ]

    combo_cols = []

    for col1, col2 in combo_pairs:
        if col1 in X.columns and col2 in X.columns:
            new_col = f"{col1}_{col2}_combo"
            X[new_col] = X[col1].astype(str) + "_" + X[col2].astype(str)
            combo_cols.append(new_col)

    return X, combo_cols


def add_oof_target_encoding(X, y, cols, n_splits=5, smoothing=10, random_state=42):
    X = X.copy().reset_index(drop=True)
    y = y.astype(int).reset_index(drop=True)

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    te_features = pd.DataFrame(index=X.index)

    for col in cols:
        te_features[f"{col}_TE"] = 0.0

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        print(f"Target Encoding Fold {fold}")

        X_tr = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_tr = y.iloc[train_idx]

        encoder = TargetEncoder(
            cols=cols,
            smoothing=smoothing
        )

        encoder.fit(X_tr[cols], y_tr)
        encoded_val = encoder.transform(X_val[cols])

        for col in cols:
            te_features.loc[val_idx, f"{col}_TE"] = encoded_val[col].values

    final_encoder = TargetEncoder(
        cols=cols,
        smoothing=smoothing
    )

    final_encoder.fit(X[cols], y)

    X_te = pd.concat([X, te_features], axis=1)

    return X_te, final_encoder

In [4]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
submission = pd.read_csv(SUBMISSION_PATH)

X_raw = train_raw.drop(columns=[TARGET])
y = train_raw[TARGET].astype(int).reset_index(drop=True)
X_test_raw = test_raw.copy()

id_cols = [col for col in X_raw.columns if "ID" in col.upper()]

X_raw = X_raw.drop(columns=id_cols, errors="ignore")
X_test_raw = X_test_raw.drop(columns=id_cols, errors="ignore")

# 기존 champion preprocessing 사용
X = data_preprocessing(X_raw)
X_test = data_preprocessing(X_test_raw)

X_combo, combo_cols = add_combo_columns(X)
X_test_combo, _ = add_combo_columns(X_test)

base_te_cols = [
    "시술 시기 코드",
    "시술 유형",
    "특정 시술 유형",
    "배란 유도 유형",
    "난자 출처",
    "정자 출처",
    "배아 생성 주요 이유",
    "시술 당시 나이",
]

base_te_cols = [col for col in base_te_cols if col in X_combo.columns]
te_cols = base_te_cols + combo_cols

print("base_te_cols:", base_te_cols)
print("combo_cols:", combo_cols)
print("te_cols count:", len(te_cols))

X_te_combo, te_encoder_combo = add_oof_target_encoding(
    X_combo,
    y,
    cols=te_cols,
    n_splits=N_SPLITS,
    smoothing=10,
    random_state=FOLD_SEED
)

test_te_values = te_encoder_combo.transform(X_test_combo[te_cols])

for col in te_cols:
    X_test_combo[f"{col}_TE"] = test_te_values[col].values

# combo 문자열 원본 제거
X_te_combo = X_te_combo.drop(columns=combo_cols, errors="ignore")
X_test_te_combo = X_test_combo.drop(columns=combo_cols, errors="ignore")

# 컬럼 정렬
X_test_te_combo = X_test_te_combo[X_te_combo.columns]

X_stack = X_te_combo.copy().reset_index(drop=True)
X_test_stack = X_test_te_combo.copy().reset_index(drop=True)
y_stack = y.reset_index(drop=True)

print("X_stack:", X_stack.shape)
print("X_test_stack:", X_test_stack.shape)
print("y_stack:", y_stack.shape)

assert list(X_stack.columns) == list(X_test_stack.columns)
assert len(X_stack) == len(y_stack)
assert len(X_test_stack) == len(submission)

base_te_cols: ['시술 시기 코드', '시술 유형', '특정 시술 유형', '배란 유도 유형', '난자 출처', '정자 출처', '배아 생성 주요 이유', '시술 당시 나이']
combo_cols: ['시술 당시 나이_난자 출처_combo', '시술 당시 나이_정자 출처_combo', '시술 당시 나이_시술 유형_combo', '시술 당시 나이_특정 시술 유형_combo', '시술 유형_난자 출처_combo', '시술 유형_정자 출처_combo', '특정 시술 유형_난자 출처_combo', '특정 시술 유형_정자 출처_combo', '난자 출처_정자 출처_combo', '배란 유도 유형_시술 당시 나이_combo']
te_cols count: 18
Target Encoding Fold 1
Target Encoding Fold 2
Target Encoding Fold 3
Target Encoding Fold 4
Target Encoding Fold 5
X_stack: (256351, 110)
X_test_stack: (90067, 110)
y_stack: (256351,)


In [5]:
def train_lgbm_oof_test(
    X_stack,
    X_test_stack,
    y_stack,
    n_splits=5,
    fold_seed=42,
    save_dir=None,
):
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=fold_seed
    )

    numeric_features = X_stack.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X_stack.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    print("numeric_features:", len(numeric_features))
    print("categorical_features:", len(categorical_features))

    # LGBM은 스케일링 필요 없음.
    # 범주형은 OneHot으로 변환해서 XGB와 다른 representation을 만들되, sparse 유지.
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", "passthrough", numeric_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )

    oof = np.zeros(len(X_stack))
    test_pred = np.zeros(len(X_test_stack))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
        print(f"\n================ LGBM Fold {fold} ================")

        X_tr = X_stack.iloc[tr_idx].copy()
        X_val = X_stack.iloc[val_idx].copy()
        y_tr = y_stack.iloc[tr_idx]
        y_val = y_stack.iloc[val_idx]

        X_tr_trans = preprocessor.fit_transform(X_tr)
        X_val_trans = preprocessor.transform(X_val)
        X_test_trans = preprocessor.transform(X_test_stack)

        model = LGBMClassifier(
            n_estimators=5000,
            learning_rate=0.015,
            num_leaves=64,
            max_depth=-1,
            min_child_samples=80,
            subsample=0.85,
            subsample_freq=1,
            colsample_bytree=0.85,
            reg_alpha=0.5,
            reg_lambda=8.0,
            objective="binary",
            random_state=fold_seed,
            n_jobs=-1,
            scale_pos_weight=POS_WEIGHT,
            verbose=-1,
        )

        model.fit(
            X_tr_trans,
            y_tr,
            eval_set=[(X_val_trans, y_val)],
            eval_metric="auc",
            callbacks=[
                lgb.early_stopping(stopping_rounds=150, verbose=True),
                lgb.log_evaluation(period=100),
            ],
        )

        val_pred = model.predict_proba(X_val_trans)[:, 1]
        oof[val_idx] = val_pred

        fold_auc = roc_auc_score(y_val, val_pred)
        fold_scores.append(fold_auc)

        print(f"LGBM Fold {fold} AUC:", fold_auc)
        print("best_iteration:", model.best_iteration_)

        test_pred += model.predict_proba(X_test_trans)[:, 1] / n_splits

        if save_dir is not None:
            np.save(save_dir / f"lgbm_partial_oof_fold{fold}.npy", oof)
            np.save(save_dir / f"lgbm_partial_test_fold{fold}.npy", test_pred)

    oof_auc = roc_auc_score(y_stack, oof)

    print("\n================ LGBM RESULT ================")
    print("Fold scores:", fold_scores)
    print("Mean fold AUC:", np.mean(fold_scores))
    print("OOF AUC:", oof_auc)

    return oof, test_pred, fold_scores

In [6]:
lgbm_oof, lgbm_test_pred, lgbm_fold_scores = train_lgbm_oof_test(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    n_splits=N_SPLITS,
    fold_seed=FOLD_SEED,
    save_dir=SAVE_DIR,
)

np.save(SAVE_DIR / "lgbm_oof.npy", lgbm_oof)
np.save(SAVE_DIR / "lgbm_test_pred.npy", lgbm_test_pred)
np.save(SAVE_DIR / "y_stack.npy", y_stack.to_numpy())

summary_df = pd.DataFrame([{
    "model": "lgbm",
    "feature_set": "combo_te_v1_s10",
    "fold_seed": FOLD_SEED,
    "oof_auc": roc_auc_score(y_stack, lgbm_oof),
    "fold_scores": str(lgbm_fold_scores),
}])

summary_df.to_csv(SAVE_DIR / "summary.csv", index=False)
display(summary_df)

numeric_features: 56
categorical_features: 54

================ LGBM Fold 1 ================
Training until validation scores don't improve for 150 rounds
[100]	valid_0's auc: 0.735771	valid_0's binary_logloss: 0.552687
Early stopping, best iteration is:
[39]	valid_0's auc: 0.733888	valid_0's binary_logloss: 0.534109
LGBM Fold 1 AUC: 0.7338878510257969
best_iteration: 39

================ LGBM Fold 2 ================


/mnt/c/dev/my_ml_project/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/mnt/c/dev/my_ml_project/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Training until validation scores don't improve for 150 rounds
[100]	valid_0's auc: 0.73859	valid_0's binary_logloss: 0.551449
Early stopping, best iteration is:
[39]	valid_0's auc: 0.736781	valid_0's binary_logloss: 0.533375
LGBM Fold 2 AUC: 0.7367805845878848
best_iteration: 39

================ LGBM Fold 3 ================


/mnt/c/dev/my_ml_project/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/mnt/c/dev/my_ml_project/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Training until validation scores don't improve for 150 rounds
[100]	valid_0's auc: 0.737426	valid_0's binary_logloss: 0.552896
Early stopping, best iteration is:
[38]	valid_0's auc: 0.735395	valid_0's binary_logloss: 0.53418
LGBM Fold 3 AUC: 0.7353948587895797
best_iteration: 38

================ LGBM Fold 4 ================


...

Training until validation scores don't improve for 150 rounds
[100]	valid_0's auc: 0.736655	valid_0's binary_logloss: 0.551613
Early stopping, best iteration is:
[39]	valid_0's auc: 0.735201	valid_0's binary_logloss: 0.5334
LGBM Fold 4 AUC: 0.7352005785168256
best_iteration: 39

================ LGBM Fold 5 ================


/mnt/c/dev/my_ml_project/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/mnt/c/dev/my_ml_project/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Training until validation scores don't improve for 150 rounds
[100]	valid_0's auc: 0.73748	valid_0's binary_logloss: 0.552585
Early stopping, best iteration is:
[37]	valid_0's auc: 0.73506	valid_0's binary_logloss: 0.533959
LGBM Fold 5 AUC: 0.735059983333714
best_iteration: 37

================ LGBM RESULT ================
Fold scores: [0.7338878510257969, 0.7367805845878848, 0.7353948587895797, 0.7352005785168256, 0.735059983333714]
Mean fold AUC: 0.7352647712507602
OOF AUC: 0.7349969202283624


/mnt/c/dev/my_ml_project/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/mnt/c/dev/my_ml_project/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,model,feature_set,fold_seed,oof_auc,fold_scores
0,lgbm,combo_te_v1_s10,42,0.734997,"[0.7338878510257969, 0.7367805845878848, 0.735..."


In [7]:
# =========================================================
# Load LGBM
# =========================================================
lgbm_oof = np.load(SAVE_DIR / "lgbm_oof.npy")
lgbm_test_pred = np.load(SAVE_DIR / "lgbm_test_pred.npy")

print("LGBM OOF:", roc_auc_score(y_stack, lgbm_oof))
print("lgbm_oof:", lgbm_oof.shape)
print("lgbm_test_pred:", lgbm_test_pred.shape)

LGBM OOF: 0.7349969202283624
lgbm_oof: (256351,)
lgbm_test_pred: (90067,)


In [8]:
# =========================================================
# Load current champion 4-seed avg OOF
# =========================================================
seed42_oof = np.load(PROJECT_ROOT / "oof_preds" / "combo_te_s10" / "final_combo_rank.npy")
seed2024_oof = np.load(PROJECT_ROOT / "oof_preds" / "combo_te_v1_s10_seed2024" / "final_oof_seed2024.npy")
seed777_oof = np.load(PROJECT_ROOT / "oof_preds" / "combo_te_v1_s10_seed777" / "final_oof_seed777.npy")
seed999_oof = np.load(PROJECT_ROOT / "oof_preds" / "combo_te_v1_s10_seed999" / "final_oof_seed999.npy")

champion_4seed_oof = (
    seed42_oof +
    seed2024_oof +
    seed777_oof +
    seed999_oof
) / 4

champion_4seed_score = roc_auc_score(y_stack, champion_4seed_oof)

print("Champion 4-seed OOF:", champion_4seed_score)

Champion 4-seed OOF: 0.7407705074934163


In [9]:
# =========================================================
# Load current champion 4-seed avg test pred
# =========================================================
seed42_test = np.load(PROJECT_ROOT / "submissions" / "preds" / "final_aggressive_pred_lb_0_74172.npy")
seed2024_test = np.load(PROJECT_ROOT / "submissions" / "seed_ensemble" / "final_pred_combo_te_v1_seed2024.npy")
seed777_test = np.load(PROJECT_ROOT / "submissions" / "seed_ensemble" / "final_pred_combo_te_v1_seed777.npy")
seed999_test = np.load(PROJECT_ROOT / "submissions" / "seed_ensemble" / "final_pred_combo_te_v1_seed999.npy")

champion_4seed_test = (
    seed42_test +
    seed2024_test +
    seed777_test +
    seed999_test
) / 4

print("champion_4seed_test:", champion_4seed_test.shape)

champion_4seed_test: (90067,)


In [10]:
champion_rank_oof = rank01(champion_4seed_oof)
lgbm_rank_oof = rank01(lgbm_oof)

best_score = 0
best_w_lgbm = None
best_blend_oof = None

for w_lgbm in np.arange(0.00, 0.31, 0.01):
    w_champ = 1 - w_lgbm

    blend_oof = (
        w_champ * champion_rank_oof +
        w_lgbm * lgbm_rank_oof
    )

    score = roc_auc_score(y_stack, blend_oof)

    if score > best_score:
        best_score = score
        best_w_lgbm = w_lgbm
        best_blend_oof = blend_oof.copy()

print("Champion 4-seed OOF:", champion_4seed_score)
print("LGBM OOF:", roc_auc_score(y_stack, lgbm_oof))
print("Best blend OOF:", best_score)
print("Best LGBM weight:", best_w_lgbm)
print("Improvement:", best_score - champion_4seed_score)

Champion 4-seed OOF: 0.7407705074934163
LGBM OOF: 0.7349969202283624
Best blend OOF: 0.7407705074934163
Best LGBM weight: 0.0
Improvement: 0.0


In [11]:
if best_score > champion_4seed_score:
    champion_rank_test = rank01(champion_4seed_test)
    lgbm_rank_test = rank01(lgbm_test_pred)

    final_pred_champion_lgbm = (
        (1 - best_w_lgbm) * champion_rank_test +
        best_w_lgbm * lgbm_rank_test
    )

    submission_blend = pd.read_csv(SUBMISSION_PATH)
    pred_col = submission_blend.columns[-1]
    submission_blend[pred_col] = final_pred_champion_lgbm

    out_csv = LGBM_SUB_DIR / f"submission_champion4seed_lgbm_w{best_w_lgbm:.2f}.csv"
    out_npy = LGBM_SUB_DIR / f"final_pred_champion4seed_lgbm_w{best_w_lgbm:.2f}.npy"

    submission_blend.to_csv(out_csv, index=False)
    np.save(out_npy, final_pred_champion_lgbm)

    np.save(SAVE_DIR / f"champion4seed_lgbm_blend_oof_w{best_w_lgbm:.2f}.npy", best_blend_oof)

    summary_blend = pd.DataFrame([{
        "champion_4seed_oof": champion_4seed_score,
        "lgbm_oof": roc_auc_score(y_stack, lgbm_oof),
        "blend_oof": best_score,
        "w_champion": 1 - best_w_lgbm,
        "w_lgbm": best_w_lgbm,
        "improvement": best_score - champion_4seed_score,
        "file": out_csv.name,
    }])

    summary_blend.to_csv(
        LGBM_SUB_DIR / f"summary_champion4seed_lgbm_w{best_w_lgbm:.2f}.csv",
        index=False
    )

    print("saved csv:", out_csv)
    print("saved npy:", out_npy)
    print(submission_blend[pred_col].describe())

else:
    print("LGBM blend did not improve OOF. Submission not saved.")

LGBM blend did not improve OOF. Submission not saved.


In [ ]:
print("SAVE_DIR files")
for p in SAVE_DIR.glob("*"):
    print(p.name)

print("\nLGBM_SUB_DIR files")
for p in LGBM_SUB_DIR.glob("*"):
    print(p.name)